# Entrenamiento del modelo · Números mayas con CNN

Proyecto 02 · Inteligencia Artificial · UVG Campus Sur

Este notebook **solo orquesta**: todo el código vive en `ml/src/maya/` del repositorio.
Nada de lógica nueva se escribe aquí, porque lo que no está versionado no existe.

**Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno → **GPU**.

| Paso | Qué hace | Tiempo aprox. con GPU |
|---|---|---|
| 1 | Clonar e instalar | 2 min |
| 2 | Generar el dataset sintético | 15 min |
| 3 | Subir las fotos reales | 3 min |
| 4 | Splits y cache | 5 min |
| 5 | Entrenar etapa A (sintético) | 20 min |
| 6 | Entrenar etapa B (afinado con reales) | 5 min |
| 7 | Evaluar y calibrar | 2 min |
| 8 | Exportar a ONNX y descargar | 1 min |


## 1 · Clonar el repositorio e instalar


In [ ]:
!git clone https://github.com/josejulianamadogarcia10-lab/reconocimiento-numeros-mayas.git repo
%cd repo/ml
!pip install -q -r requirements.txt
import torch; print('GPU disponible:', torch.cuda.is_available(), '|', torch.__version__)


## 2 · Generar el dataset sintético

Unas 25 000 imágenes. Es el grueso del volumen de datos: la variación de grosor, fondo,
iluminación y perspectiva es lo que hace que el modelo tolere una foto real.


In [ ]:
import sys; sys.path.insert(0, 'src')
from maya import synth
n = synth.generar(raiz_datos='../data', por_clase=800, negativos=10000, semilla=42)
print(f'{n} imágenes sintéticas')


## 3 · Subir las fotos reales

Comprimí `data/raw/real/` en un zip y subilo acá. Si todavía no hay fotos, saltá esta celda:
el modelo se entrena igual con sintético, solo que generalizará peor a fotos de verdad.


In [ ]:
from google.colab import files
import zipfile, pathlib
subido = files.upload()          # elegí el zip de data/raw/real
for nombre in subido:
    with zipfile.ZipFile(nombre) as z:
        z.extractall('../data/raw/')
print('fotos reales:', len(list(pathlib.Path('../data/raw/real').rglob('*.jpg'))))


## 4 · Splits y cache

`--autor-prueba` es el integrante que queda **fuera** del entrenamiento: sus fotos son el
conjunto de prueba. Es lo más parecido a la mano desconocida del docente el sábado.

La verificación anti-fuga es obligatoria: si falla, no se entrena.


In [ ]:
!python -m maya.dataset --raiz ../data --construir-splits --autor-prueba imri
!python -m maya.dataset --raiz ../data --verificar
!python -m maya.dataset --raiz ../data --cache


## 5 · Etapa A — entrenar con el sintético


In [ ]:
!python -m maya.train --config configs/default.yaml --etapa A


## 6 · Etapa B — afinar con las fotos reales

Poné abajo el id de la corrida que imprimió la celda anterior. La etapa B mezcla las fotos
reales repetidas con parte del sintético: sin esa mezcla el modelo olvida lo aprendido.


In [ ]:
RUN_A = 'runs/XXXXXXXX-XXXXXX-A'   # <-- pegá acá el id de la etapa A
!python -m maya.train --config configs/default.yaml --etapa B --desde {RUN_A}


## 7 · Calibrar y evaluar

La calibración ajusta la temperatura y busca los umbrales maximizando la utilidad con los
pesos de la rúbrica (ADR-006). La evaluación toca el conjunto de prueba **una sola vez**.


In [ ]:
RUN = 'runs/XXXXXXXX-XXXXXX-B'    # <-- la corrida final
!python -m maya.calibrate --run {RUN} --raiz ../data
!python -m maya.evaluate  --run {RUN} --raiz ../data --split test


In [ ]:
from IPython.display import Image, display
for fig in ('curvas.png', 'confusion.png', 'por_clase.png'):
    display(Image(f'{RUN}/{fig}'))


## 8 · Exportar a ONNX y descargar

El export verifica que el ONNX dé lo mismo que PyTorch (max|diff| < 1e-5 y 100% de
coincidencia en la etiqueta). Si no se cumple, no escribe nada.


In [ ]:
!python -m maya.export_onnx --run {RUN} --destino ../web/public/models --raiz ../data
!cd .. && zip -r modelo.zip web/public/models {RUN}/metricas.json {RUN}/*.png
from google.colab import files
files.download('../modelo.zip')


---
**Después:** descomprimí `modelo.zip` en el repositorio, commiteá `web/public/models/` y
actualizá `docs/ESTADO.md` con las métricas reales. Ningún número de la presentación puede
salir de otro lugar que no sea el `metricas.json` de una corrida.
